# Inspect DuckDB from Hugging Face
### Database: `20260201_20260228_M1_EURUSDm.duckdb`
### Dataset Repository: [`Sarkaravijit123/my-tst-data`](https://huggingface.co/datasets/Sarkaravijit123/my-tst-data)

This notebook downloads `20260201_20260228_M1_EURUSDm.duckdb` directly from Hugging Face Datasets,
inspects all tables (`ohlcv`, `mt5_trades`, `vbt_trades`, `vbt_metrics`, `performance_summary`),
and verifies trade reconciliation parity between MetaTrader 5 and VectorBT.

In [ ]:
# 1. Setup & Dependencies
# If running in Google Colab or fresh environment, uncomment:
# !pip install -q huggingface_hub duckdb pandas

import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

In [ ]:
# 2. Download DuckDB file from Hugging Face Hub
REPO_ID = "Sarkaravijit123/my-tst-data"
FILENAME = "20260201_20260228_M1_EURUSDm.duckdb"

print(f"Downloading {FILENAME} from https://huggingface.co/datasets/{REPO_ID}...")
db_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    repo_type="dataset"
)
print(f"Cached local snapshot at: {db_path}")

In [ ]:
# 3. Connect to DuckDB and List Tables with Row Counts
con = duckdb.connect(db_path, read_only=True)

tables_df = con.execute("SHOW TABLES").df()
summary = []
for tbl in tables_df["name"]:
    cnt = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    summary.append({"table_name": tbl, "total_rows": cnt})

pd.DataFrame(summary)

--- 
## 4. Inspect `ohlcv` (1-Minute Candles with Spread)

In [ ]:
# Date range and metrics
ohlcv_meta = con.execute("""
    SELECT 
        MIN(timestamp) AS start_time,
        MAX(timestamp) AS end_time,
        COUNT(*) AS total_bars,
        ROUND(AVG(spread), 1) AS avg_spread,
        MIN(low) AS min_low,
        MAX(high) AS max_high
    FROM ohlcv
""").df()
display(ohlcv_meta)

# First 5 bars
con.execute("SELECT * FROM ohlcv ORDER BY timestamp ASC LIMIT 5").df()

--- 
## 5. Inspect `mt5_trades` (MetaTrader 5 Strategy Tester)

In [ ]:
# MT5 Summary Metrics
mt5_summary = con.execute("""
    SELECT 
        COUNT(*) AS total_trades,
        SUM(CASE WHEN profit > 0 THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN profit <= 0 THEN 1 ELSE 0 END) AS losses,
        ROUND(AVG(CASE WHEN profit > 0 THEN 1.0 ELSE 0.0 END) * 100, 2) AS win_rate_pct,
        ROUND(SUM(profit), 2) AS net_pnl,
        ROUND(AVG(profit), 2) AS avg_trade_pnl
    FROM mt5_trades
""").df()
display(mt5_summary)

# Preview MT5 trades
con.execute("""
    SELECT trade_id, entry_time, exit_time, entry_price, exit_price, volume, profit, comment_in, comment_out 
    FROM mt5_trades 
    ORDER BY entry_time ASC 
    LIMIT 10
""").df()

--- 
## 6. Inspect `vbt_trades` (VectorBT Simulation)

In [ ]:
# VBT Summary Metrics
vbt_summary = con.execute("""
    SELECT 
        COUNT(*) AS total_trades,
        SUM(CASE WHEN pnl > 0 THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN pnl <= 0 THEN 1 ELSE 0 END) AS losses,
        ROUND(AVG(CASE WHEN pnl > 0 THEN 1.0 ELSE 0.0 END) * 100, 2) AS win_rate_pct,
        ROUND(SUM(pnl), 2) AS net_pnl,
        ROUND(AVG(pnl), 2) AS avg_trade_pnl
    FROM vbt_trades
""").df()
display(vbt_summary)

# Preview VBT trades
con.execute("""
    SELECT trade_id, entry_time, exit_time, entry_price, exit_price, active_sl, tp_price, sl_mode, exit_reason, pnl 
    FROM vbt_trades 
    ORDER BY entry_time ASC 
    LIMIT 10
""").df()

--- 
## 7. Trade-by-Trade Timestamp Alignment (MT5 vs VectorBT)

In [ ]:
m = con.execute("SELECT trade_id AS mt5_id, entry_time, exit_time, entry_price AS m_in, exit_price AS m_out, profit AS mt5_profit, comment_out FROM mt5_trades ORDER BY entry_time ASC").df()
v = con.execute("SELECT trade_id AS vbt_id, entry_time, exit_time, entry_price AS v_in, exit_price AS v_out, pnl AS vbt_pnl, exit_reason FROM vbt_trades ORDER BY entry_time ASC").df()

m["entry_dt"] = pd.to_datetime(m["entry_time"])
m["exit_dt"] = pd.to_datetime(m["exit_time"])
v["entry_dt"] = pd.to_datetime(v["entry_time"], utc=True).dt.tz_localize(None)
v["exit_dt"] = pd.to_datetime(v["exit_time"], utc=True).dt.tz_localize(None)

matched_rows = []
used_v = set()

for _, mr in m.iterrows():
    v_avail = v[~v.index.isin(used_v)]
    if v_avail.empty: break
    diffs = (v_avail["entry_dt"] - mr["entry_dt"]).abs().dt.total_seconds()
    valid = diffs[diffs <= 60]
    if not valid.empty:
        best_v_idx = valid.idxmin()
        vr = v.loc[best_v_idx]
        used_v.add(best_v_idx)
        entry_diff = int(diffs.loc[best_v_idx])
        exit_diff = int(abs((vr["exit_dt"] - mr["exit_dt"]).total_seconds()))
        matched_rows.append({
            "mt5_id": mr["mt5_id"],
            "vbt_id": vr["vbt_id"],
            "entry_time": mr["entry_dt"],
            "entry_delta_sec": entry_diff,
            "mt5_exit": mr["exit_dt"],
            "vbt_exit": vr["exit_dt"],
            "exit_delta_sec": exit_diff,
            "mt5_profit": mr["mt5_profit"],
            "vbt_pnl": vr["vbt_pnl"],
            "mt5_comm": mr["comment_out"],
            "vbt_reason": vr["exit_reason"],
        })

df_matched = pd.DataFrame(matched_rows)
exact_entries = sum(df_matched["entry_delta_sec"] == 0)
aligned_both = sum((df_matched["entry_delta_sec"] == 0) & (df_matched["exit_delta_sec"] <= 120))

print(f"Total MT5 trades: {len(m)}")
print(f"Total VectorBT trades: {len(v)}")
print(f"Matched entries: {len(df_matched)} / {len(m)}")
print(f"Exact 0s entry delta: {exact_entries} / {len(m)} ({exact_entries / len(m) * 100:.1f}%)")
print(f"Execution-aligned (<=120s): {aligned_both} / {len(m)} ({aligned_both / len(m) * 100:.1f}%)")

display(df_matched.head(15))

In [ ]:
# 8. Close Connection
con.close()
print("DuckDB connection cleanly closed.")